In [ ]:
import json
import pandas as pd
from IPython.display import HTML, display

with open("gold_standard_annotations_duplicate_ext.json") as f:
    results = json.load(f)

df_sample = pd.read_csv("gold_standard_sample.csv.gz", compression="gzip")
print(f"Loaded: {len(df_sample)} judgments")

Loaded: 1000 judgments


In [ ]:
def annotatie_editor(ecli, results, df_sample):
    result = next(r for r in results if r["ecli"] == ecli)
    tekst = df_sample[df_sample["ecli"] == ecli]["text"].iloc[0]
    
    bestaande = json.dumps(result["annotations"])
    tekst_escaped = tekst.replace("\\", "\\\\").replace("`", "\\`").replace("${", "\\${")
    
    html = f"""
    <div style="font-family: sans-serif; padding: 1rem 0;">
      <h3 style="margin-bottom: 0.5rem;">{ecli}</h3>
      
      <div style="display: flex; gap: 8px; margin-bottom: 1rem; flex-wrap: wrap; align-items: center;">
        <button id="btn-BWB"   style="padding: 4px 12px; border-radius: 20px; border: 2px solid #f0a500; background: #fff176; color: #7a5200; font-size: 13px; cursor: pointer;">BWB</button>
        <button id="btn-ECLI"  style="padding: 4px 12px; border-radius: 20px; border: 2px solid #4a90d9; background: #90caf9; color: #0d47a1; font-size: 13px; cursor: pointer;">ECLI</button>
        <button id="btn-CELEX" style="padding: 4px 12px; border-radius: 20px; border: 2px solid #66bb6a; background: #a5d6a7; color: #1b5e20; font-size: 13px; cursor: pointer;">CELEX</button>
        <button id="btn-OEP"   style="padding: 4px 12px; border-radius: 20px; border: 2px solid #ffa040; background: #ffcc80; color: #7a3b00; font-size: 13px; cursor: pointer;">OEP</button>
        <span style="font-size: 13px; color: gray;">Actief: <strong id="active-label">—</strong></span>
      </div>

      <div style="font-size: 12px; color: gray; margin-bottom: 0.5rem;">
        Selecteer een label, daarna tekst om te annoteren. Klik op een span om te verwijderen.
      </div>

      <div id="text-container" style="line-height: 2.2; font-size: 14px; border: 1px solid #ddd; border-radius: 8px; padding: 1rem; max-height: 500px; overflow-y: auto; background: white; color: black;"></div>

      <div style="margin-top: 1rem; display: flex; gap: 8px;">
        <button id="btn-export" style="padding: 6px 14px; cursor: pointer;">Export JSON</button>
        <button id="btn-clear" style="padding: 6px 14px; cursor: pointer; color: red;">Reset naar origineel</button>
      </div>

      <div id="stats" style="margin-top: 0.5rem; font-size: 13px; color: gray;"></div>

      <div id="export-area" style="display:none; margin-top: 1rem;">
        <div style="font-size: 12px; color: gray; margin-bottom: 4px;">Kopieer deze JSON terug in je notebook:</div>
        <textarea id="json-out" style="width: 100%; height: 150px; font-family: monospace; font-size: 12px; border: 1px solid #ddd; border-radius: 4px; padding: 8px;" readonly></textarea>
      </div>
    </div>

    <script>
    (function() {{
      const LABELS = {{
        BWB:   {{ bg: '#fff176', border: '#f0a500', color: '#7a5200' }},
        ECLI:  {{ bg: '#90caf9', border: '#4a90d9', color: '#0d47a1' }},
        CELEX: {{ bg: '#a5d6a7', border: '#66bb6a', color: '#1b5e20' }},
        OEP:   {{ bg: '#ffcc80', border: '#ffa040', color: '#7a3b00' }}
      }};

      let activeLabel = null;
      let annotations = {bestaande};
      const plainText = `{tekst_escaped}`;

      function esc(str) {{
        return str.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');
      }}

      function render() {{
        const container = document.getElementById('text-container');
        const sorted = [...annotations].sort((a, b) => a.start - b.start);
        let html = '';
        let pos = 0;
        for (const ann of sorted) {{
          if (ann.start < pos) continue;
          html += esc(plainText.slice(pos, ann.start));
          const c = LABELS[ann.label] || {{ bg: '#eee', border: '#999', color: '#333' }};
          const isDup = ann.is_duplicate ? '★ ' : '';
          html += `<mark data-start="${{ann.start}}" data-end="${{ann.end}}" style="background:${{c.bg}};border-bottom:2px solid ${{c.border}};color:${{c.color}};cursor:pointer;border-radius:3px;padding:1px 3px;" title="${{isDup}}${{ann.label}} — klik om te verwijderen">${{esc(plainText.slice(ann.start, ann.end))}}<sup style="font-size:10px;margin-left:2px;">${{ann.label}}</sup></mark>`;
          pos = ann.end;
        }}
        html += esc(plainText.slice(pos));
        container.innerHTML = html;

        // Event listeners op marks
        container.querySelectorAll('mark').forEach(mark => {{
          mark.addEventListener('click', function() {{
            const start = parseInt(this.dataset.start);
            const end = parseInt(this.dataset.end);
            annotations = annotations.filter(a => !(a.start === start && a.end === end));
            render();
          }});
        }});

        updateStats();
      }}

      function updateStats() {{
        const counts = {{}};
        annotations.forEach(a => counts[a.label] = (counts[a.label]||0) + 1);
        const parts = Object.entries(counts).map(([l,n]) => `${{l}}: ${{n}}`);
        document.getElementById('stats').textContent = annotations.length
          ? `${{annotations.length}} spans — ${{parts.join(', ')}}`
          : '';
      }}

      // Label buttons
      Object.keys(LABELS).forEach(label => {{
        document.getElementById('btn-' + label).addEventListener('click', function() {{
          activeLabel = label;
          document.getElementById('active-label').textContent = label;
          Object.keys(LABELS).forEach(l => {{
            document.getElementById('btn-' + l).style.fontWeight = l === label ? '700' : '400';
            document.getElementById('btn-' + l).style.boxShadow = l === label ? '0 0 0 3px ' + LABELS[l].border : 'none';
          }});
        }});
      }});

      // Export
      document.getElementById('btn-export').addEventListener('click', function() {{
        const out = annotations.map(a => ({{
          span: a.span, label: a.label, start: a.start, end: a.end
        }}));
        document.getElementById('json-out').value = JSON.stringify(out, null, 2);
        document.getElementById('export-area').style.display = 'block';
      }});

      // Originele annotaties bewaren
      const origineleAnnotations = JSON.parse(JSON.stringify(annotations));

      // Reset naar origineel
      document.getElementById('btn-clear').addEventListener('click', function() {{
        annotations = JSON.parse(JSON.stringify(origineleAnnotations));
        render();
      }});

      // Tekst selecteren om span toe te voegen
      document.getElementById('text-container').addEventListener('mouseup', function() {{
        if (!activeLabel) return;
        const sel = window.getSelection();
        if (!sel || sel.isCollapsed) return;
        const range = sel.getRangeAt(0);
        const container = document.getElementById('text-container');
        const preRange = document.createRange();
        preRange.selectNodeContents(container);
        preRange.setEnd(range.startContainer, range.startOffset);
        const start = preRange.toString().length;
        const span = sel.toString();
        if (!span.trim()) {{ sel.removeAllRanges(); return; }}
        const end = start + span.length;
        const overlaps = annotations.some(a => !(end <= a.start || start >= a.end));
        if (overlaps) {{ sel.removeAllRanges(); return; }}
        annotations.push({{
          span: plainText.slice(start, end),
          label: activeLabel,
          start, end,
          is_duplicate: false
        }});
        sel.removeAllRanges();
        render();
      }});

      render();
    }})();
    </script>
    """
    
    return HTML(html)

In [ ]:
ecli = df_sample['ecli'].iloc[0]

'ECLI:NL:RBDHA:2025:8235'

In [33]:
ecli = df_sample['ecli'].iloc[11]

display(annotatie_editor(ecli, results, df_sample))